In [1]:
!date

Fri Feb 20 14:18:31 CET 2026


In [2]:
pwd

'/Users/aidasaglinskas/Desktop/ANDSpeak/Google_drive_things/Code-2026'

In [31]:
import pandas as pd

import base64
from openai import OpenAI
import os
from tqdm import tqdm
import numpy as np

In [2]:
def diarized_to_csv(transcription, out_csv=None):
    """
    Convert OpenAI diarized transcription object to CSV.

    Parameters
    ----------
    transcription : TranscriptionDiarized
        Object returned by OpenAI API
    out_csv : str
        Path to output CSV
    """

    rows = []

    for seg in transcription.segments:
        rows.append({
            "segment_id": seg.id,
            "speaker": seg.speaker,
            "start_sec": seg.start,
            "end_sec": seg.end,
            "duration_sec": seg.end - seg.start,
            "text": seg.text.strip()
        })

    df = pd.DataFrame(rows)

    # useful ordering guarantee
    df = df.sort_values("start_sec").reset_index(drop=True)

    if out_csv:
        df.to_csv(out_csv, index=False)

    return df

In [36]:
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [37]:
df = pd.read_csv('../DementiaBank-preprocessed2/file_list.csv')
df

,Unnamed: 0.1,Unnamed: 0,path,site,group,filename,diag,subName,size_MB,dur_s,dur_m
0,0,0,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,033-1.mp3,D,Pitt_D_033,2.38,155.83,2.597167
1,1,1,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,033-0.mp3,D,Pitt_D_033,0.85,55.58,0.926333
2,2,2,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,076-0.mp3,D,Pitt_D_076,1.25,81.83,1.363833
3,3,3,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,049-0.mp3,D,Pitt_D_049,1.07,70.18,1.169667
4,4,4,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,076-2.mp3,D,Pitt_D_076,1.45,95.13,1.585500
...,...,...,...,...,...,...,...,...,...,...,...
280,280,56,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,33-1.mp3,D,Delaware_D_33,6.50,425.59,7.093167
281,281,57,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,15-2.mp3,D,Delaware_D_15,8.86,857.43,14.290500
282,282,58,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,69-1.mp3,D,Delaware_D_69,4.25,278.69,4.644833
283,283,59,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,11-1.mp3,D,Delaware_D_11,15.24,998.52,16.642000


In [38]:
def get_transcript(audio_path):
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="gpt-4o-transcribe-diarize",
            file=audio_file,
            response_format="diarized_json",
            chunking_strategy="auto",
            language="en",
            temperature=0)
    return transcript

In [39]:
idx = np.random.permutation(np.arange(len(df)))
for i in tqdm(idx):
    path = df['path'].values[i]
    ofdir = '../DementiaBank-preprocessed2/01-diarized-transcripts-v1/'
    ofn = df['filename'].values[i].replace('.mp3','.csv')
    out_filename = os.path.join(ofdir,ofn)
    
    if not os.path.exists(out_filename):
        transcript = get_transcript(path)
        diarized_csv = diarized_to_csv(transcript,out_filename)

 13%|███████████▊                                                                               | 37/285 [1:00:09<6:43:16, 97.57s/it]


InternalServerError: Error code: 500 - {'error': {'message': 'The server had an error processing your request. Sorry about that! You can retry your request, or contact us through our help center at help.openai.com if you keep seeing this error. (Please include the request ID req_425ab074a764456cb132e42e29b3cd88 in your email.)', 'type': 'server_error', 'param': None, 'code': None}}